# CS4100 Final Project  
Team Members: Khushi Khan, Dustin Zhang, Kayla Handley, Koena Gupta

In [1]:
import pandas as pd
import re
from pandas.api.types import is_numeric_dtype, is_string_dtype

In [12]:
# Load datasets
df_cleaned = pd.read_csv('../../data/cleaned/spotify_master_clean.csv', low_memory=False)

print(df_cleaned.shape)

(109926, 19)


In [13]:
df_cleaned.head()

,track_name,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,instrumentalness,mode,key,duration_ms,acousticness,source,artist_name,popularity,track_name_clean,artist_name_clean
0,!!!!,0.258,0.440074,0.758032,0.744005,0.109109,0.0593,0.8,0.087785,0.890,0.0,1.000000,0.024918,0.515060,nov_2018,alicks,0.24,NaN,alicks
1,!!!!!!!,0.278,0.000000,0.000000,0.620814,0.669670,0.0000,0.0,0.000000,0.000,1.0,0.090909,0.001850,0.771084,april_2019,billie eilish,0.33,NaN,billie eilish
2,"""1955",0.437,0.328034,0.317269,0.776171,0.090591,0.0393,0.6,0.037371,0.901,1.0,0.818182,0.033294,0.632530,nov_2018,pablo lópez,0.00,1955,pablo lópez
3,"""42"" - from sr3mm",0.563,0.520191,0.971888,0.861664,0.108108,0.3240,0.8,0.129400,0.000,1.0,0.090909,0.041881,0.002761,nov_2018,rae sremmurd,0.55,42 from sr3mm,rae sremmurd
4,"""99""",0.804,0.383946,0.554217,0.901223,0.111111,0.7140,0.8,0.031366,0.000,1.0,0.727273,0.034995,0.006004,nov_2018,barns courtney,0.64,99,barns courtney


In [47]:
import numpy as np

# this maps valence/arousal to four different emotion categories: "joy/excitement", "peaceful/content", "anger/tension", and "sadness"
# each category gets a score from 0 (least amount of that emotion) to 1 (most amount of that emotion)
# - (valence, energy) = (1, 1) ==> maximum joy, minimum sadness
# - (valence, energy) = (1, 0) ==> maximum peace, minimum anger
# - (valence, energy) = (0, 1) ==> maximum anger, minimum peace
# - (valence, energy) = (0, 0) ==> maximum sadness, minimum joy
# currently, a song's score for each category is the distance of its (valence, energy) from the opposite emotion normalized to [0, 1]
# so if a song has (valence, energy) = (0.3, 0.4) then its joy score is sqrt((0.3 - 0)^2 + (0.4 - 0)^2) / sqrt(2) = 0.5 / 1.414 = 0.354
# since the opposite of joy is sadness, which corresponds to (valence, energy) = (0, 0)
# tbh I'm not sure if this is the best way of assigning emotion labels, I'll see if there are better ways in the literature somewhere

valence = df_cleaned['valence']
arousal = df_cleaned['energy']
df_cleaned['emotion_joy_excitement'] = np.sqrt(valence ** 2 + arousal ** 2) / np.sqrt(2)
df_cleaned['emotion_peaceful_content'] = np.sqrt(valence ** 2 + (1 - arousal) ** 2) / np.sqrt(2)
df_cleaned['emotion_anger_tension'] = np.sqrt((1 - valence) ** 2 + arousal ** 2) / np.sqrt(2)
df_cleaned['emotion_sadness'] = np.sqrt((1 - valence) ** 2 + (1 - arousal) ** 2) / np.sqrt(2)

# this is a softmax of the four emotion scores (so they all sum to 1)
# idk if this is necessary but it might make building the model a bit easier since we can use a softmax output layer
# and we don't have to worry about making sure the outputs are properly normalized

emotion_sum = np.exp(df_cleaned['emotion_joy_excitement']) \
    + np.exp(df_cleaned['emotion_peaceful_content']) \
          + np.exp(df_cleaned['emotion_anger_tension']) \
              + np.exp(df_cleaned['emotion_sadness'])

df_cleaned['emotion_joy_excitement_softmax'] = np.exp(df_cleaned['emotion_joy_excitement']) / emotion_sum
df_cleaned['emotion_peaceful_content_softmax'] = np.exp(df_cleaned['emotion_peaceful_content']) / emotion_sum
df_cleaned['emotion_anger_tension_softmax'] = np.exp(df_cleaned['emotion_anger_tension']) / emotion_sum
df_cleaned['emotion_sadness_softmax'] = np.exp(df_cleaned['emotion_sadness']) / emotion_sum
df_cleaned.head()

,track_name,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,instrumentalness,...,track_name_clean,artist_name_clean,emotion_joy_excitement,emotion_peaceful_content,emotion_anger_tension,emotion_sadness,emotion_joy_excitement_softmax,emotion_peaceful_content_softmax,emotion_anger_tension_softmax,emotion_sadness_softmax
0,!!!!,0.258,0.440074,0.758032,0.744005,0.109109,0.0593,0.8,0.087785,0.890,...,NaN,alicks,0.187190,0.526346,0.689739,0.847196,0.166903,0.234293,0.275879,0.322925
1,!!!!!!!,0.278,0.000000,0.000000,0.620814,0.669670,0.0000,0.0,0.000000,0.000,...,NaN,billie eilish,0.196576,0.510531,0.733922,0.872148,0.165414,0.226423,0.283099,0.325064
2,"""1955",0.437,0.328034,0.317269,0.776171,0.090591,0.0393,0.6,0.037371,0.901,...,1955,pablo lópez,0.310253,0.399070,0.746295,0.787373,0.190444,0.208133,0.294536,0.306887
3,"""42"" - from sr3mm",0.563,0.520191,0.971888,0.861664,0.108108,0.3240,0.8,0.129400,0.000,...,42 from sr3mm,rae sremmurd,0.459317,0.384672,0.622071,0.569186,0.236913,0.219873,0.278787,0.264427
4,"""99""",0.804,0.383946,0.554217,0.901223,0.111111,0.7140,0.8,0.031366,0.000,...,99,barns courtney,0.760333,0.523551,0.603412,0.245165,0.308515,0.243469,0.263710,0.184307


In [48]:
df_cleaned.describe()

,energy,tempo,danceability,loudness,liveness,valence,time_signature,speechiness,instrumentalness,mode,...,acousticness,popularity,emotion_joy_excitement,emotion_peaceful_content,emotion_anger_tension,emotion_sadness,emotion_joy_excitement_softmax,emotion_peaceful_content_softmax,emotion_anger_tension_softmax,emotion_sadness_softmax
count,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,...,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000,109926.000000
mean,0.563684,0.476453,0.578174,0.804780,0.197555,0.444226,0.774436,0.115216,0.235821,0.610875,...,0.356527,0.248196,0.530558,0.486379,0.602543,0.524218,0.250144,0.236977,0.264937,0.247942
std,0.266308,0.120768,0.194395,0.110491,0.171134,0.263125,0.104925,0.129288,0.367041,0.487554,...,0.353333,0.183827,0.214764,0.165429,0.142589,0.211898,0.050564,0.037115,0.033806,0.049392
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.007469,0.004123,0.128625,0.128625,0.130087,0.129459
25%,0.382000,0.382862,0.450803,0.771612,0.097497,0.223000,0.800000,0.040269,0.000000,0.000000,...,0.033434,0.110000,0.388038,0.374217,0.517064,0.363465,0.216321,0.212769,0.244796,0.210100
50%,0.602000,0.480085,0.601406,0.839579,0.124124,0.426000,0.800000,0.057143,0.000212,1.000000,...,0.217871,0.230000,0.563298,0.493642,0.613725,0.518139,0.258697,0.240239,0.265172,0.247045
75%,0.777000,0.553522,0.727912,0.878167,0.241241,0.649000,0.800000,0.130435,0.537000,1.000000,...,0.672691,0.360000,0.696334,0.616065,0.689129,0.677047,0.289306,0.263072,0.287254,0.284245
max,1.000000,1.000000,1.000000,0.999094,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,0.996001,0.993305,1.000000,1.000000,0.349059,0.348640,0.349640,0.349640


In [49]:
# Writing processed dataset to a CSV file
df_cleaned.to_csv("../../data/cleaned/spotify_master_clean_emotion_labels.csv", index=False)